# Real-World Simulation: Watch Reflexio Learn

> **Time:** ~20 minutes | **Level:** Advanced

This notebook simulates a realistic production scenario. You'll watch
Reflexio learn from a flawed conversation, then see a subsequent conversation
improve using the learned context.

```
┌──────────────┐    publish    ┌──────────────┐
│ Conversation │──────────────→│   Reflexio   │
│ (User↔Agent) │               │              │
└──────────────┘               │  Extracts:   │
      ↑                        │  - Profiles  │
      │                        │  - Playbooks │
      │     retrieve context   └──────┬───────┘
      └───────────────────────────────┘
```

In [ ]:
import uuid

import pandas as pd
from _display_helpers import *
from IPython.display import Markdown, display
from rich import print as rprint

from reflexio import InteractionData, ReflexioClient
from reflexio.models.config_schema import (
    PlaybookAggregatorConfig,
    PlaybookConfig,
    ProfileExtractorConfig,
)

url, api_key = load_env()
client = ReflexioClient(url_endpoint=url, api_key=api_key)
RUN_ID = uuid.uuid4().hex[:8]
USER_ID = f"customer_mei_{RUN_ID}"
print(f"Run ID: {RUN_ID}")
print(f"User ID: {USER_ID}")

## Step 1: Configure Reflexio for Customer Support

> Set extraction stride to 1 so every interaction triggers extraction.

In [ ]:
config = client.get_config()

config.profile_extractor_configs = [
    ProfileExtractorConfig(
        extractor_name="customer_info",
        profile_content_definition_prompt=(
            "Extract key user information: name, dietary restrictions, "
            "allergies, preferences, account details, or any personal facts."
        ),
        context_prompt="Restaurant customer support conversation.",
        batch_interval_override=1,
        batch_size_override=20,
    )
]

config.playbook_configs = [
    PlaybookConfig(
        playbook_name="agent_quality",
        playbook_definition_prompt=(
            "Identify cases where the agent made mistakes, missed user needs, "
            "gave incorrect info, or could have handled the situation better."
        ),
        playbook_aggregator_config=PlaybookAggregatorConfig(min_cluster_size=1),
        batch_interval_override=1,
        batch_size_override=20,
    )
]

config.batch_interval = 1
client.set_config(config)
show_success("Configured Reflexio for customer support")

In [ ]:
# Verify the configuration
show_config(client.get_config())

## Step 2: The Baseline Conversation (No Memory)

> The agent handles a customer **without any prior knowledge**. The customer has a severe peanut allergy, but the agent doesn't know this yet.

In [ ]:
baseline_conversation = [
    InteractionData(
        role="User",
        content="Hi! I'd like to order the pad thai, please.",
    ),
    InteractionData(
        role="Agent",
        content=(
            "Great choice! Our pad thai is very popular. "
            "Would you like mild, medium, or spicy?"
        ),
    ),
    InteractionData(
        role="User",
        content=(
            "Medium, please. Oh wait \u2014 does it have peanuts? "
            "I have a severe peanut allergy."
        ),
    ),
    InteractionData(
        role="Agent",
        content=(
            "Let me check... Yes, our pad thai is made with peanut sauce "
            "and crushed peanuts on top."
        ),
    ),
    InteractionData(
        role="User",
        content=(
            "I really wish you had mentioned that upfront! "
            "That could be dangerous for me. What else do you have that's peanut-free?"
        ),
    ),
    InteractionData(
        role="Agent",
        content=(
            "I'm so sorry about that! Let me suggest some peanut-free options: "
            "our green curry, tom yum soup, or mango sticky rice are all peanut-free."
        ),
    ),
]

In [ ]:
show_interactions(baseline_conversation, title="Baseline Conversation (No Memory)")

## Step 3: Publish to Reflexio

> Publishing triggers automatic extraction. Reflexio will learn about the customer AND identify the agent's mistake.

In [ ]:
client.publish_interaction(
    user_id=USER_ID,
    interactions=baseline_conversation,
    source="notebook",
    session_id=f"baseline_{RUN_ID}",
    wait_for_response=True,
)
show_success("Published! Reflexio is analyzing the conversation...")

## Step 4: See What Reflexio Learned

> Reflexio extracted **two types** of insights:
> 1. **User profiles** — facts about the customer (peanut allergy)
> 2. **Agent playbooks** — what the agent should do differently

In [ ]:
display(Markdown("#### User Profiles"))
profiles = client.get_profiles(force_refresh=True, user_id=USER_ID)
show_profiles(profiles.user_profiles)

In [ ]:
display(Markdown("#### Agent Playbooks"))
playbooks = client.get_user_playbooks()
show_playbooks(playbooks.user_playbooks)

In [ ]:
# Peek at the raw extracted data for deeper understanding
if playbooks.user_playbooks:
    pb = playbooks.user_playbooks[0]
    display(Markdown("#### User Playbook Detail"))
    show_json(pb, title=f"Playbook: {pb.playbook_name}")

## Step 5: Build an Enhanced Prompt

> Now the same customer returns. Before responding, we **retrieve** Reflexio context and inject it into the agent's prompt.

In [ ]:
# Retrieve relevant context
profile_resp = client.search_profiles(user_id=USER_ID, query="ordering food", top_k=5)
playbook_resp = client.search_user_playbooks(query="customer ordering food", top_k=5)

rprint(f"Profiles found: [bold]{len(profile_resp.user_profiles)}[/bold]")
rprint(f"Playbooks found: [bold]{len(playbook_resp.user_playbooks)}[/bold]")

In [ ]:
profile_context = "\n".join(f"- {p.profile_content}" for p in profile_resp.user_profiles)

playbook_lines = []
for pb in playbook_resp.user_playbooks:
    if hasattr(pb, "content") and pb.content:
        playbook_lines.append(f"- {pb.content}")
playbook_context = "\n".join(playbook_lines)

enhanced_prompt = f"""You are a restaurant support agent.

## Known about this customer:
{profile_context or "No information yet."}

## Behavior corrections from past interactions:
{playbook_context or "No corrections yet."}

Respond to the customer's message below.
"""

display(Markdown("### Enhanced Agent Prompt"))
print(enhanced_prompt)

## Step 6: The Improved Conversation

> With Reflexio context, the agent **proactively** warns about the peanut allergy.

In [ ]:
enhanced_conversation = [
    InteractionData(
        role="User",
        content="Hi! I'd like to order the pad thai, please.",
    ),
    InteractionData(
        role="Agent",
        content=(
            "Welcome back! Before I put that order in \u2014 I see you have a severe "
            "peanut allergy on file, and our pad thai contains peanut sauce and "
            "crushed peanuts. Can I suggest some delicious peanut-free alternatives "
            "instead? Our green curry and tom yum soup are both excellent and "
            "completely peanut-free!"
        ),
    ),
    InteractionData(
        role="User",
        content=(
            "Oh wow, thank you for catching that! "
            "The green curry sounds great, I'll go with that."
        ),
    ),
    InteractionData(
        role="Agent",
        content=(
            "Excellent choice! One green curry coming right up. "
            "I've double-checked \u2014 it's completely peanut-free. "
            "Would you like mild, medium, or spicy?"
        ),
    ),
]

In [ ]:
show_interactions(enhanced_conversation, title="Enhanced Conversation (With Memory)")

## Side-by-Side Comparison

In [ ]:
# Compare how the agent responded to the same opening message
comparison = pd.DataFrame([
    {
        "Version": "Baseline (No Memory)",
        "Agent Response to Pad Thai Order": baseline_conversation[1].content,
        "Allergy Mentioned": "No \u2014 customer had to ask",
    },
    {
        "Version": "Enhanced (With Memory)",
        "Agent Response to Pad Thai Order": enhanced_conversation[1].content,
        "Allergy Mentioned": "Yes \u2014 proactively warned",
    },
])

display(Markdown("### Baseline vs Enhanced"))
display(comparison.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

## The Improvement Loop

> Each conversation makes the agent smarter:
> 1. **Profiles** accumulate user preferences (allergies, favorites)
> 2. **Playbooks** capture agent mistakes (forgot to warn about allergens)
> 3. Future conversations benefit from **all past learning**

In [ ]:
# Publish the enhanced conversation too
client.publish_interaction(
    user_id=USER_ID,
    interactions=enhanced_conversation,
    source="notebook",
    session_id=f"enhanced_{RUN_ID}",
    wait_for_response=True,
)

# Show updated counts
profiles = client.get_profiles(force_refresh=True, user_id=USER_ID)
playbooks = client.get_user_playbooks()
rprint(f"Total profiles: [bold]{len(profiles.user_profiles)}[/bold]")
rprint(f"Total playbooks: [bold]{len(playbooks.user_playbooks)}[/bold]")

In [ ]:
# See the full picture after both conversations
display(Markdown("### All Profiles After Two Conversations"))
show_profiles(profiles.user_profiles)

display(Markdown("### All Playbooks After Two Conversations"))
show_playbooks(playbooks.user_playbooks)

## (Optional) Try Your Own Scenario

> Modify the conversations below to see Reflexio learn from your own examples!

In [ ]:
# Template \u2014 customize these interactions:
# my_conversation = [
#     InteractionData(role="User", content="Your message here..."),
#     InteractionData(role="Agent", content="Agent response here..."),
# ]
# client.publish_interaction(
#     user_id=USER_ID,
#     interactions=my_conversation,
#     source="notebook",
#     session_id=f"custom_{RUN_ID}",
#     wait_for_response=True,
# )

In [ ]:
# --- Cleanup: reset config and remove all data ---
try:
    config = client.get_config()
    config.profile_extractor_configs = []
    config.playbook_configs = []
    config.batch_interval = 5
    client.set_config(config)
except Exception:
    pass

client.delete_all_interactions()
client.delete_all_profiles()
client.delete_all_playbooks()
show_success("Config reset and data cleaned up.")

## Summary

Reflexio works behind the scenes: **publish interactions** \u2192 profiles and playbooks **auto-extract** \u2192 **retrieve context** \u2192 agent **improves**.

The core loop is simple:
1. Conversations happen naturally
2. Reflexio extracts user facts (profiles) and agent mistakes (playbooks)
3. Before the next conversation, retrieve relevant context
4. Inject it into the agent's prompt — the agent is now smarter

| Notebook | What you'll learn |
|----------|-------------------|
| [00 — Quickstart](00_quickstart.ipynb) | Your first Reflexio workflow |
| [05 — Concurrent Sessions](05_concurrent_sessions.ipynb) | Multi-user load and data isolation |